# Deep Dive: Prompt Instruction Refinement

## 📋 Summary

**Prompt Instruction Refinement** is the systematic process of improving a prompt until it reliably
produces the output you need. It is the closest thing to "prompt engineering as a discipline".

### The core cycle
```
Draft prompt → Run it → Evaluate output → Identify weakness → Refine → Repeat
```

### Why structured refinement matters
A vague prompt gives the model too much freedom. Each refinement step *constrains the solution space*:
- **Role** tells the model who it is
- **Context** gives it what it needs to know
- **Task** defines what to do
- **Format** defines how to structure the answer
- **Examples** show exactly what "good" looks like
- **Constraints** rule out what you don't want
- **Evaluation criteria** help you know when you're done

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Vague vs. specific prompts | 🟢 Beginner |
| 2 | Adding components one at a time | 🟢 Beginner |
| 3 | Few-shot examples | 🟡 Intermediate |
| 4 | Negative examples & constraints | 🟡 Intermediate |
| 5 | Systematic evaluation with a rubric | 🔴 Advanced |
| 6 | Exercises | 🏋️ Practice |


## 0. Setup

In [1]:
# key input
import os
from google.colab import userdata
# set the key on colab on the left "key" icon (SECRETS)
# Automatically configures the key for SDKs to grab behind the scenes
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [2]:
# Install / import
from openai import OpenAI
from IPython.display import Markdown, display
import json, re

# ── API key ──────────────────────────────────────────────────────────────────

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"] ,
)

MODEL = "gpt-4.1-nano"

def chat(system: str, user: str, temperature: float = 0.3) -> str:
    """Single-turn helper. Returns the assistant text."""
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content

def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))
    return Markdown(f"**{label}**\n\n{text}\n\n---").data

def compare(labels_and_prompts: list[tuple[str, str, str]], user: str):
    """Run multiple system prompts against the same user message and display side by side."""
    for label, system, _ in labels_and_prompts:
        r = chat(system, user)
        show(label, r)

## 1. Vague vs. Specific Prompts 🟢

The gap between a vague and a specific prompt is often enormous.
Let's use a concrete task: **writing a product description for an e-commerce listing**.


In [3]:
product_info = """
Product: Ergonomic Mesh Office Chair
Features: lumbar support, adjustable armrests, breathable mesh back, 5-year warranty
Price: $349
Target buyer: remote workers with back pain
"""

# V0: Vague
v0_system = "Write product descriptions."
r0 = chat(v0_system, f"Product info:\n{product_info}")
show("V0 — Vague", r0)


**V0 — Vague**

Experience unmatched comfort with our Ergonomic Mesh Office Chair, designed specifically for remote workers seeking relief from back pain. Featuring superior lumbar support and adjustable armrests, this chair ensures personalized comfort to keep you focused and pain-free throughout your workday. The breathable mesh back promotes airflow, keeping you cool even during long hours. Built with durability in mind, it comes with a 5-year warranty for peace of mind. Elevate your home office setup for just $349 and enjoy ergonomic excellence every day.

---

'**V0 — Vague**\n\nExperience unmatched comfort with our Ergonomic Mesh Office Chair, designed specifically for remote workers seeking relief from back pain. Featuring superior lumbar support and adjustable armrests, this chair ensures personalized comfort to keep you focused and pain-free throughout your workday. The breathable mesh back promotes airflow, keeping you cool even during long hours. Built with durability in mind, it comes with a 5-year warranty for peace of mind. Elevate your home office setup for just $349 and enjoy ergonomic excellence every day.\n\n---'

In [4]:
# V1: Specific task + role
v1_system = """You are a senior copywriter specializing in e-commerce product pages.
Write a compelling product description for the following item."""
r1 = chat(v1_system, f"Product info:\n{product_info}")
show("V1 — Role + Task", r1)


**V1 — Role + Task**

Experience the ultimate in comfort and support with our Ergonomic Mesh Office Chair, designed specifically for remote workers who value health and productivity. Crafted with a breathable mesh back, this chair ensures optimal airflow to keep you cool during long work sessions. The built-in lumbar support promotes proper posture, alleviating back pain and reducing fatigue. Customize your comfort with adjustable armrests that adapt to your preferred position, allowing you to work effortlessly for hours on end. Built to last with premium materials and backed by a 5-year warranty, this chair is the perfect investment in your well-being and work efficiency. Elevate your home office setup today for a healthier, more comfortable workday at just $349.

---

'**V1 — Role + Task**\n\nExperience the ultimate in comfort and support with our Ergonomic Mesh Office Chair, designed specifically for remote workers who value health and productivity. Crafted with a breathable mesh back, this chair ensures optimal airflow to keep you cool during long work sessions. The built-in lumbar support promotes proper posture, alleviating back pain and reducing fatigue. Customize your comfort with adjustable armrests that adapt to your preferred position, allowing you to work effortlessly for hours on end. Built to last with premium materials and backed by a 5-year warranty, this chair is the perfect investment in your well-being and work efficiency. Elevate your home office setup today for a healthier, more comfortable workday at just $349.\n\n---'

### 🧐 What changed?
- Length? Tone? Structure?
- Did V1 address the target buyer (remote workers with back pain)?


## 2. Adding Components One at a Time 🟢

We'll iterate from V1 → V2 → V3 by adding one prompt component per step
and seeing the cumulative effect.


In [5]:
# V2: Add output format
v2_system = """You are a senior copywriter specializing in e-commerce product pages.

Write a compelling product description for the following item.

Output format:
- Headline (max 10 words, benefit-focused)
- Opening hook (1–2 sentences, addresses the pain point)
- 3–5 bullet points of key features (start each with an action verb)
- Closing CTA (1 sentence)"""

r2 = chat(v2_system, f"Product info:\n{product_info}")
show("V2 — Role + Task + Format", r2)


**V2 — Role + Task + Format**

Ultimate Comfort for Remote Workers with Back Pain

Say goodbye to discomfort and hello to productivity with our Ergonomic Mesh Office Chair, designed to support your back and keep you comfortable during long work hours.

- Provide targeted lumbar support to reduce back strain and improve posture  
- Adjust armrests effortlessly to customize your arm and shoulder comfort  
- Enhance airflow with a breathable mesh back that keeps you cool all day  
- Built to last with a durable design backed by a 5-year warranty  
- Elevate your home office setup for maximum comfort and focus  

Upgrade your workspace today and experience the difference—your back will thank you!

---

'**V2 — Role + Task + Format**\n\nUltimate Comfort for Remote Workers with Back Pain\n\nSay goodbye to discomfort and hello to productivity with our Ergonomic Mesh Office Chair, designed to support your back and keep you comfortable during long work hours.\n\n- Provide targeted lumbar support to reduce back strain and improve posture  \n- Adjust armrests effortlessly to customize your arm and shoulder comfort  \n- Enhance airflow with a breathable mesh back that keeps you cool all day  \n- Built to last with a durable design backed by a 5-year warranty  \n- Elevate your home office setup for maximum comfort and focus  \n\nUpgrade your workspace today and experience the difference—your back will thank you!\n\n---'

In [6]:
# V3: Add context + constraints
v3_system = """You are a senior copywriter specializing in e-commerce product pages.

Context: This description will appear on Amazon. Competitors are priced $200–$500.
The main buyer is a remote worker who sits 8+ hours a day and has chronic back pain.

Write a compelling product description for the following item.

Constraints:
- Total length: 150–200 words
- Avoid superlatives like "best", "greatest", "number one"
- Do NOT mention the price
- Use second-person "you" to speak directly to the buyer

Output format:
- Headline (max 10 words, benefit-focused)
- Opening hook (1–2 sentences, addresses the pain point)
- 3–5 bullet points (start each with an action verb)
- Closing CTA (1 sentence)"""

r3 = chat(v3_system, f"Product info:\n{product_info}")
show("V3 — Full structured prompt", r3)


**V3 — Full structured prompt**

Experience Comfort That Supports Your Day

If you spend hours at your desk, lingering discomfort and back pain can take a toll. This ergonomic mesh office chair is designed to keep you comfortable and focused, so you can work longer without strain.

- **Align your posture** with adjustable lumbar support that adapts to your back’s natural curve.
- **Customize your fit** by adjusting armrests to reduce shoulder tension and promote proper arm positioning.
- **Stay cool and comfortable** with a breathable mesh back that promotes airflow during long work sessions.
- **Enjoy peace of mind** with a durable 5-year warranty, ensuring lasting support and quality.
- **Enhance your workspace** with a sleek, modern design that complements any home office setup.

Take control of your comfort—upgrade your workspace today and experience the difference in your daily productivity and well-being.

---

'**V3 — Full structured prompt**\n\nExperience Comfort That Supports Your Day\n\nIf you spend hours at your desk, lingering discomfort and back pain can take a toll. This ergonomic mesh office chair is designed to keep you comfortable and focused, so you can work longer without strain.\n\n- **Align your posture** with adjustable lumbar support that adapts to your back’s natural curve.\n- **Customize your fit** by adjusting armrests to reduce shoulder tension and promote proper arm positioning.\n- **Stay cool and comfortable** with a breathable mesh back that promotes airflow during long work sessions.\n- **Enjoy peace of mind** with a durable 5-year warranty, ensuring lasting support and quality.\n- **Enhance your workspace** with a sleek, modern design that complements any home office setup.\n\nTake control of your comfort—upgrade your workspace today and experience the difference in your daily productivity and well-being.\n\n---'

### 🧐 Prompt Component Tracker

| Component | V0 | V1 | V2 | V3 |
|---|---|---|---|---|
| Role | ❌ | ✅ | ✅ | ✅ |
| Context | ❌ | ❌ | ❌ | ✅ |
| Task | ✅ | ✅ | ✅ | ✅ |
| Output format | ❌ | ❌ | ✅ | ✅ |
| Constraints | ❌ | ❌ | ❌ | ✅ |

Which version would you use in production and why?


## 3. Few-Shot Examples 🟡

Examples are the most powerful prompt component because they show — not tell — what
"good" looks like. We'll add a worked example to our prompt and watch the quality jump.

### When to use few-shot examples
- When the output format is complex or unusual
- When the model keeps misinterpreting instructions
- When you need consistent style across many runs


In [7]:
EXAMPLE_INPUT = """
Product: Portable Laptop Stand
Features: aluminum, adjustable height, foldable, fits 11–17 inch laptops
Target buyer: digital nomads and frequent travelers
"""

EXAMPLE_OUTPUT = """
**Work From Anywhere. Without the Neck Pain.**

If you've ever hunched over a laptop in a café or airport lounge and felt your neck ache by noon,
you know the problem. This stand was built to fix it.

- **Adjusts to your ideal viewing angle** — 6 height settings from 0° to 45°
- **Carries itself** — folds flat in seconds, weighs just 580g
- **Fits every laptop you own** — compatible with 11–17 inch models
- **Built to last** — aircraft-grade aluminum survives daily travel abuse

Take it everywhere. Set up in 10 seconds. Your spine will thank you.
"""

v4_system = f"""You are a senior copywriter specializing in e-commerce product pages.

Context: Listings appear on Amazon. Main buyer is a remote worker with chronic back pain.

Write a compelling product description for the following item.

Constraints:
- 150–200 words total
- No superlatives ("best", "greatest", "number one")
- Do NOT mention the price
- Use second-person "you"

Output format:
- Headline (max 10 words)
- Opening hook (1–2 sentences)
- 3–5 bullet points (action verb start)
- Closing CTA

Here is an example of an ideal output for a different product:

INPUT:
{EXAMPLE_INPUT}

OUTPUT:
{EXAMPLE_OUTPUT}
"""

r4 = chat(v4_system, f"Product info:\n{product_info}")
show("V4 — With Few-Shot Example", r4)


**V4 — With Few-Shot Example**

Comfort That Supports Your Day

If long hours at your desk leave your back sore and stiff, this chair is designed to help you sit better and feel more comfortable throughout your workday.

- **Relieve pressure with adjustable lumbar support** — customize to fit your lower back for all-day comfort
- **Enhance your posture** — adjustable armrests allow you to find the perfect arm position
- **Stay cool and focused** — breathable mesh back promotes airflow to prevent overheating
- **Ensure lasting quality** — backed by a 5-year warranty for peace of mind

Transform your workspace into a place where you can work comfortably and stay focused. Upgrade your chair today.

---

'**V4 — With Few-Shot Example**\n\nComfort That Supports Your Day\n\nIf long hours at your desk leave your back sore and stiff, this chair is designed to help you sit better and feel more comfortable throughout your workday.\n\n- **Relieve pressure with adjustable lumbar support** — customize to fit your lower back for all-day comfort\n- **Enhance your posture** — adjustable armrests allow you to find the perfect arm position\n- **Stay cool and focused** — breathable mesh back promotes airflow to prevent overheating\n- **Ensure lasting quality** — backed by a 5-year warranty for peace of mind\n\nTransform your workspace into a place where you can work comfortably and stay focused. Upgrade your chair today.\n\n---'

## 4. Negative Examples & Constraints 🟡

Sometimes it's easier to say what you *don't* want. Negative examples and explicit
constraints prevent common failure modes.


In [8]:
# The same task but with negative examples added
BAD_EXAMPLE = """
**The Best Chair You'll Ever Sit In!**

This is truly the greatest office chair on the market, bar none. Featuring incredible lumbar
support that is second to none, this amazing product will revolutionize your sitting experience.
Buy it today and discover why thousands call it the #1 chair!

- Amazing support
- Great quality
- Wonderful warranty
"""

v5_system = v4_system + f"""

Here is an example of a BAD output to avoid:

BAD OUTPUT (do not produce this):
{BAD_EXAMPLE}

Why it's bad:
- Uses superlatives ("best", "greatest", "#1")
- Vague bullet points with no specifics
- Hype over substance
- Doesn't address buyer pain points
"""

r5 = chat(v5_system, f"Product info:\n{product_info}")
show("V5 — With Negative Example", r5)


**V5 — With Negative Example**

Comfort That Supports Your Day

If back pain slows you down during long work hours, this ergonomic mesh office chair is designed to help you stay comfortable and focused.

- **Provides targeted lumbar support** — encourages proper posture to reduce strain
- **Adjusts easily** — customizable armrests and seat height for your perfect fit
- **Keeps you cool** — breathable mesh back promotes airflow and prevents overheating
- **Built to last** — backed by a 5-year warranty for peace of mind

Make your workspace more comfortable. Sit smarter and feel better every day.

---

'**V5 — With Negative Example**\n\nComfort That Supports Your Day\n\nIf back pain slows you down during long work hours, this ergonomic mesh office chair is designed to help you stay comfortable and focused.\n\n- **Provides targeted lumbar support** — encourages proper posture to reduce strain\n- **Adjusts easily** — customizable armrests and seat height for your perfect fit\n- **Keeps you cool** — breathable mesh back promotes airflow and prevents overheating\n- **Built to last** — backed by a 5-year warranty for peace of mind\n\nMake your workspace more comfortable. Sit smarter and feel better every day.\n\n---'

### 🧐 Did the negative example help?
- Are the bullet points more specific?
- Are superlatives avoided?


## 5. Systematic Evaluation with a Rubric 🔴

Instead of judging outputs by eye, we can ask a *second model call* to evaluate
each output against a rubric. This is the foundation of **automated prompt evaluation**.


In [9]:
EVALUATOR_SYSTEM = """You are an expert e-commerce copywriting evaluator.
Grade the following product description on these criteria (0–10 each):

1. Pain-point relevance: Does it speak directly to the target buyer's problem?
2. Specificity: Are the benefits concrete and backed by product facts?
3. Guideline compliance: Does it avoid superlatives and omit the price?
4. Format adherence: Does it follow headline → hook → bullets → CTA?
5. CTA strength: Is the call to action compelling?

Output ONLY valid JSON:
{
  "pain_point_relevance": <0-10>,
  "specificity": <0-10>,
  "guideline_compliance": <0-10>,
  "format_adherence": <0-10>,
  "cta_strength": <0-10>,
  "total": <sum>,
  "comments": "<one sentence per criterion, separated by | >"
}"""

def evaluate(description: str) -> dict:
    raw = chat(EVALUATOR_SYSTEM, f"Product description to evaluate:\n{description}", temperature=0.1)
    try:
        # Strip markdown code fences if present
        clean = raw.strip().strip('`').replace('json\n','').replace('json','',1).strip()
        return json.loads(clean)
    except Exception:
        return {"raw": raw, "error": "parse failed"}

# Evaluate all versions we've created
versions = {
    "V0 — Vague":           r0,
    "V1 — Role+Task":       r1,
    "V2 — +Format":         r2,
    "V3 — +Constraints":    r3,
    "V4 — +Few-Shot":       r4,
    "V5 — +Negative Ex.":   r5,
}

results = {}
for label, text in versions.items():
    score = evaluate(text)
    results[label] = score
    total = score.get("total", "?")
    print(f"{label:30s} → Total: {total}/50")

print("\n✅ Evaluation complete. See results dict for full breakdown.")


V0 — Vague                     → Total: 33/50
V1 — Role+Task                 → Total: 33/50
V2 — +Format                   → Total: 39/50
V3 — +Constraints              → Total: 39/50
V4 — +Few-Shot                 → Total: 39/50
V5 — +Negative Ex.             → Total: 40/50

✅ Evaluation complete. See results dict for full breakdown.


In [10]:
# Show the full evaluation for the best and worst version
import json

best_label  = max(results, key=lambda k: results[k].get("total", 0))
worst_label = min(results, key=lambda k: results[k].get("total", 0))

print(f"Best:  {best_label}")
print(json.dumps(results[best_label], indent=2))
print(f"\nWorst: {worst_label}")
print(json.dumps(results[worst_label], indent=2))


Best:  V5 — +Negative Ex.
{
  "pain_point_relevance": 8,
  "specificity": 8,
  "guideline_compliance": 9,
  "format_adherence": 8,
  "cta_strength": 7,
  "total": 40,
  "comments": "Addresses back pain and comfort issues relevant to office workers | Provides specific features like lumbar support and breathable mesh backed by product facts | Avoids superlatives and price mention | Follows a clear structure with headline, benefits, and a closing statement | The CTA is somewhat compelling but could be more direct or urgent"
}

Worst: V0 — Vague
{
  "pain_point_relevance": 8,
  "specificity": 7,
  "guideline_compliance": 6,
  "format_adherence": 7,
  "cta_strength": 5,
  "total": 33,
  "comments": "It addresses back pain relief and comfort for remote workers | It mentions features like lumbar support and breathable mesh but lacks detailed product facts | It includes a price, which should be omitted, and uses superlatives like 'unmatched' | It has a clear headline and benefits but lacks a d

## 6. Exercises 🏋️


### Exercise 1 — Identify the Weakest Component
Take the V3 system prompt and intentionally remove one component at a time.
For each removal, run the prompt and note what breaks. Which single component
makes the biggest difference?


In [ ]:
# TODO: Create v3_no_format (remove output format) and evaluate
v3_no_format = """
[Paste V3 here but remove the output format section]
"""

# r_no_format = chat(v3_no_format, f"Product info:\n{product_info}")
# show("V3 without format", r_no_format)
# score = evaluate(r_no_format)
# print(score)


In [18]:
v3_context =""" This description will appear on Amazon. Competitors are priced $200–$500.
The main buyer is a remote worker who sits 8+ hours a day and has chronic back pain. """
v3_constraints = """ - Total length: 150–200 words
- Avoid superlatives like "best", "greatest", "number one"
- Do NOT mention the price
- Use second-person "you" to speak directly to the buyer"""
v3_output = """ - Headline (max 10 words, benefit-focused)
- Opening hook (1–2 sentences, addresses the pain point)
- 3–5 bullet points (start each with an action verb)
- Closing CTA (1 sentence)"""
v3_role = """You are a senior copywriter specializing in e-commerce product pages"""

In [25]:
# V3: Add context + constraints
v3_system_full = f"""{v3_role}

Context: {v3_context}

Write a compelling product description for the following item.

Constraints:
{v3_constraints}

Output format:
{v3_output}"""

#r3 = chat(v3_system, f"Product info:\n{product_info}")
#show("V3 — Full structured prompt", r3)
r3_full = chat(v3_system_full, f"Product info:\n{product_info}")
show("V3 - full", r3_full)
score = evaluate(r3_full)
print(score)

**V3 - full**

Relieve Back Pain with Comfort You Can Count On

If you spend hours at your desk, lingering discomfort and back pain can take a toll on your day. An ergonomic chair designed for long-term support helps you stay focused and comfortable, no matter how many hours you work.

- Experience customizable comfort with adjustable armrests tailored to your height and posture
- Support your lower back with targeted lumbar support that promotes proper spinal alignment
- Stay cool and comfortable with a breathable mesh back that encourages airflow throughout your workday
- Enjoy peace of mind with a durable design backed by a 5-year warranty

Transform your workspace into a haven of comfort and support. Invest in a chair that adapts to your needs and helps reduce back pain—your back will thank you.

---

{'pain_point_relevance': 8, 'specificity': 7, 'guideline_compliance': 9, 'format_adherence': 8, 'cta_strength': 7, 'total': 39, 'comments': 'It directly addresses back pain caused by prolonged sitting | The benefits are specific, mentioning adjustable armrests, lumbar support, and breathable mesh | It avoids superlatives and price mentions | It follows a clear structure with headline, benefits, and closing statement | The CTA is somewhat compelling but could be more urgent or action-oriented'}


In [24]:
v3_wo_context = v3_system_full.replace(v3_context, "")
r3_wo_context = chat(v3_wo_context, f"Product info:\n{product_info}")
show("V3 without context", r3_wo_context)
score = evaluate(r3_wo_context)
print(score)
v3_wo_constraints = v3_system_full.replace(v3_constraints, "")
r3_wo_constraints = chat(v3_wo_constraints, f"Product info:\n{product_info}")
show("V3 without constraints", r3_wo_constraints)
score = evaluate(r3_wo_constraints)
print(score)
v3_wo_output = v3_system_full.replace(v3_output, "")
r3_wo_output = chat(v3_wo_output, f"Product info:\n{product_info}")
show("V3 without output", r3_wo_output)
score = evaluate(r3_wo_output)
print(score)

**V3 without context**

Experience Comfort and Support All Day Long

If back pain interrupts your productivity, this ergonomic mesh office chair is designed to provide relief and keep you comfortable during long work sessions. Say goodbye to discomfort and hello to focused, pain-free workdays.

- **Supports** your lower back with adjustable lumbar support that adapts to your posture
- **Enhances** airflow with a breathable mesh back, keeping you cool and comfortable
- **Customizes** your experience with adjustable armrests for optimal arm and shoulder positioning
- **Ensures** durability and peace of mind with a 5-year warranty, protecting your investment
- **Promotes** better posture and reduces strain, helping you stay productive and pain-free

Upgrade your workspace and experience the difference that thoughtful design can make. Take the first step toward a more comfortable workday today.

---

{'pain_point_relevance': 8, 'specificity': 8, 'guideline_compliance': 9, 'format_adherence': 8, 'cta_strength': 7, 'total': 40, 'comments': 'The description directly addresses back pain and comfort issues | The benefits are specific, mentioning adjustable lumbar support, breathable mesh, and warranty | It avoids superlatives and price mention | It follows a logical structure with headline, benefits, and closing statement, but lacks a clear, separate CTA button or phrase | The CTA is somewhat compelling but could be more urgent or action-oriented'}


**V3 without constraints**

Experience All-Day Comfort with Our Ergonomic Mesh Office Chair

Say goodbye to back pain and discomfort during long work hours with our expertly designed ergonomic chair, built to support your body and boost productivity.

- **Relieve** pressure with adjustable lumbar support tailored to your spine’s natural curve  
- **Customize** your comfort by easily adjusting armrests to suit your preferred position  
- **Enhance** airflow and stay cool with a breathable mesh back that promotes ventilation  
- **Ensure** long-term peace of mind with a durable 5-year warranty  
- **Transform** your workspace into a haven of comfort and efficiency today

Upgrade your home office and experience the difference—your back will thank you.

---

{'pain_point_relevance': 8, 'specificity': 7, 'guideline_compliance': 9, 'format_adherence': 8, 'cta_strength': 7, 'total': 39, 'comments': 'It addresses back pain and discomfort effectively | The benefits are specific but could include more product facts | It avoids superlatives and price mention | It follows a clear structure with headline, bullets, and CTA | The CTA is somewhat compelling but could be more urgent or action-oriented'}


**V3 without output**

Experience all-day comfort with our Ergonomic Mesh Office Chair, designed specifically for those who spend long hours at their desk. The supportive lumbar feature helps maintain proper spine alignment, reducing strain and alleviating back pain. Adjustable armrests allow you to customize your arm position for optimal comfort, while the breathable mesh back promotes airflow, keeping you cool even during extended work sessions. Crafted with durability in mind, this chair is backed by a 5-year warranty, giving you confidence in its long-term performance. Whether you're working on a tight deadline or taking a quick break, this chair adapts to your needs, helping you stay focused and comfortable throughout the day. Invest in your well-being and create a workspace that supports your health and productivity.

---

{'pain_point_relevance': 9, 'specificity': 8, 'guideline_compliance': 9, 'format_adherence': 8, 'cta_strength': 7, 'total': 41, 'comments': 'It directly addresses back pain and discomfort from prolonged sitting | Benefits are specific, such as lumbar support and breathable mesh, backed by product features | It avoids superlatives and price mention | It mostly follows the structure but lacks a clear headline and explicit CTA | The CTA is somewhat compelling but could be more direct and urgent.'}


In [21]:
print(v3_system_full)

You are a senior copywriter specializing in e-commerce product pages

Context:  This description will appear on Amazon. Competitors are priced $200–$500.
The main buyer is a remote worker who sits 8+ hours a day and has chronic back pain. 

Write a compelling product description for the following item.

Constraints:
 - Total length: 150–200 words
- Avoid superlatives like "best", "greatest", "number one"
- Do NOT mention the price
- Use second-person "you" to speak directly to the buyer

Output format:
 - Headline (max 10 words, benefit-focused)
- Opening hook (1–2 sentences, addresses the pain point)
- 3–5 bullet points (start each with an action verb)
- Closing CTA (1 sentence)


### Exercise 2 — Refinement for a New Domain
Apply the full refinement process (V0 → V5) to a **completely different task**:
writing a tweet thread (5 tweets) about a technology topic of your choice.
Define your own rubric with 3–5 criteria.


In [26]:
# TODO: Pick a topic and a starting vague prompt
topic = "the rise of local AI models running on consumer hardware"

v0_tweet_system = "Write tweets."

# Helper function for tweet evaluation
EVALUATOR_SYSTEM_TWEET = """You are an expert social media content evaluator. Grade the following tweet thread on these criteria (0–10 each):

1.  Relevance: Is the thread relevant to the topic 'the rise of local AI models running on consumer hardware'?
2.  Engagement: Is the thread engaging and likely to capture audience interest?
3.  Clarity & Conciseness: Is each tweet clear, concise, and within typical tweet length constraints?
4.  Thread Structure: Does it flow well as a thread (e.g., using (1/5), (2/5) format, logical progression)?
5.  No Hashtag Spam: Does it avoid excessive or irrelevant hashtags?

Output ONLY valid JSON:
{
  "relevance": <0-10>,
  "engagement": <0-10>,
  "clarity_conciseness": <0-10>,
  "thread_structure": <0-10>,
  "no_hashtag_spam": <0-10>,
  "total": <sum>,
  "comments": "<one sentence per criterion, separated by | >"
}"""

def evaluate_tweet(tweet_thread: str) -> dict:
    raw = chat(EVALUATOR_SYSTEM_TWEET, f"Tweet thread to evaluate:\n{tweet_thread}", temperature=0.1)
    try:
        clean = raw.strip().strip('`').replace('json\n','').replace('json','',1).strip()
        return json.loads(clean)
    except Exception:
        return {"raw": raw, "error": "parse failed"}

# V0: Vague
r0_tweet = chat(v0_tweet_system, f"Write a 5-tweet thread about {topic}")
show("V0 — Vague Tweet", r0_tweet)
eval_r0_tweet = evaluate_tweet(r0_tweet)
print(f"V0 Tweet Score: {eval_r0_tweet.get('total', 0)}/50")

# V1: Add role + task
v1_tweet_system = """You are a tech journalist and social media manager. Your task is to write a compelling 5-tweet thread about the specified topic."""
r1_tweet = chat(v1_tweet_system, f"Topic: {topic}")
show("V1 — Role + Task Tweet", r1_tweet)
eval_r1_tweet = evaluate_tweet(r1_tweet)
print(f"V1 Tweet Score: {eval_r1_tweet.get('total', 0)}/50")

# V2: Add output format
v2_tweet_system = v1_tweet_system + """

Output Format:
- The thread must consist of exactly 5 tweets.
- Each tweet should start with (X/5), where X is the tweet number.
- Each tweet should be concise and focused.
- The last tweet should include a call to action or a thought-provoking question."""
r2_tweet = chat(v2_tweet_system, f"Topic: {topic}")
show("V2 — Role + Task + Format Tweet", r2_tweet)
eval_r2_tweet = evaluate_tweet(r2_tweet)
print(f"V2 Tweet Score: {eval_r2_tweet.get('total', 0)}/50")

# V3: Add constraints
v3_tweet_system = v2_tweet_system + """

Constraints:
- Each tweet should be under 280 characters.
- Avoid excessive use of hashtags (max 2 relevant hashtags per tweet).
- Maintain a positive and forward-looking tone.
- Focus on benefits and implications for users/developers.
- Do NOT use more than 5 tweets."""
r3_tweet = chat(v3_tweet_system, f"Topic: {topic}")
show("V3 — Role + Task + Format + Constraints Tweet", r3_tweet)
eval_r3_tweet = evaluate_tweet(r3_tweet)
print(f"V3 Tweet Score: {eval_r3_tweet.get('total', 0)}/50")

# V4: Add positive example
EXAMPLE_INPUT_TWEET = "Topic: The future of quantum computing"
EXAMPLE_OUTPUT_TWEET = """
(1/5) Quantum computing is still in its early stages, but its potential to revolutionize industries like medicine, finance, and AI is immense. #QuantumTech #FutureIsNow

(2/5) Unlike classical computers, quantum machines use superposition and entanglement to process vast amounts of data simultaneously. This opens doors to solving problems currently intractable for even supercomputers.

(3/5) Imagine designing new materials atom by atom, or simulating complex drug interactions with unprecedented accuracy. Quantum algorithms promise to unlock these capabilities.

(4/5) Challenges remain, including error correction and building stable qubits at scale. But breakthroughs are happening faster than ever, driven by global research efforts.

(5/5) What breakthroughs are you most excited to see in quantum computing? Share your thoughts below! 👇 #TechTrends #Innovation
"""

v4_tweet_system = v3_tweet_system + f"""

Here is an example of an ideal tweet thread:

INPUT:
{EXAMPLE_INPUT_TWEET}

OUTPUT:
{EXAMPLE_OUTPUT_TWEET}"""
r4_tweet = chat(v4_tweet_system, f"Topic: {topic}")
show("V4 — With Few-Shot Example Tweet", r4_tweet)
eval_r4_tweet = evaluate_tweet(r4_tweet)
print(f"V4 Tweet Score: {eval_r4_tweet.get('total', 0)}/50")

# V5: Add negative example
BAD_EXAMPLE_TWEET = """
(1/5) Local AI is amazing! So cool! #AI #Tech #Awesome #SuperCool #Innovation

(2/5) It runs on your computer. Very fast. Buy new hardware.

(3/5) This is the best technology ever. Seriously the best.

(4/5) Everybody should use local AI. It's the future. Get on board.

(5/5) End of thread.
"""

v5_tweet_system = v4_tweet_system + f"""

Here is an example of a BAD output to avoid:

BAD OUTPUT (do not produce this):
{BAD_EXAMPLE_TWEET}

Why it's bad:
- Uses excessive hashtags (spammy)
- Not informative or engaging
- Vague and lacks specifics
- Uses superlatives and hype
- Doesn't follow a clear narrative or CTA
"""
r5_tweet = chat(v5_tweet_system, f"Topic: {topic}")
show("V5 — With Negative Example Tweet", r5_tweet)
eval_r5_tweet = evaluate_tweet(r5_tweet)
print(f"V5 Tweet Score: {eval_r5_tweet.get('total', 0)}/50")

# Collect all results for comparison
tweet_versions_results = {
    "V0 — Vague Tweet": eval_r0_tweet,
    "V1 — Role + Task Tweet": eval_r1_tweet,
    "V2 — Role + Task + Format Tweet": eval_r2_tweet,
    "V3 — Role + Task + Format + Constraints Tweet": eval_r3_tweet,
    "V4 — With Few-Shot Example Tweet": eval_r4_tweet,
    "V5 — With Negative Example Tweet": eval_r5_tweet,
}

print("\n--- Tweet Refinement Summary ---")
for label, score_dict in tweet_versions_results.items():
    print(f"{label:45s} -> Total: {score_dict.get('total', 0)}/50")

best_tweet_label = max(tweet_versions_results, key=lambda k: tweet_versions_results[k].get("total", 0))
worst_tweet_label = min(tweet_versions_results, key=lambda k: tweet_versions_results[k].get("total", 0))

print(f"\nBest Tweet Version:  {best_tweet_label}")
print(json.dumps(tweet_versions_results[best_tweet_label], indent=2))
print(f"\nWorst Tweet Version: {worst_tweet_label}")
print(json.dumps(tweet_versions_results[worst_tweet_label], indent=2))


**V0 — Vague Tweet**

🧵1/5  
The landscape of AI is shifting! We're seeing a surge in local AI models that run directly on consumer hardware—no cloud needed. This democratizes access, enhances privacy, and reduces reliance on big tech. Exciting times ahead! #AI #EdgeComputing

🧵2/5  
Thanks to advancements in model compression and optimization, lightweight AI models now deliver impressive performance on laptops, desktops, and even smartphones. This means faster, more private AI experiences for everyday users. #ML #OnDeviceAI

🧵3/5  
Open-source frameworks like TensorFlow Lite and ONNX are making it easier for developers to deploy custom AI models locally. We're witnessing a boom in tailored solutions for tasks like image recognition, language processing, and more. #OpenSource #AIModels

🧵4/5  
Privacy concerns are a big driver—running models locally means sensitive data stays on your device. Plus, reduced latency improves real-time applications like AR, voice assistants, and gaming. The future is truly decentralized! #Privacy #EdgeAI

🧵5/5  
As hardware continues to improve and models become more efficient, expect a wave of innovative local AI tools that empower users and developers alike. The era of AI on consumer devices is just beginning! 🚀 #FutureOfAI #AIRevolution

---

V0 Tweet Score: 45/50


**V1 — Role + Task Tweet**

🚀 The future is here! We're witnessing a surge in local AI models running directly on consumer hardware. No more relying solely on cloud servers—powerful AI is now in your hands, on your device. Here's why this shift matters: 🧵👇 #AI #TechInnovation

1/ Traditionally, AI models needed massive cloud infrastructure, making access expensive and privacy a concern. Now, with advancements in model compression and optimization, smaller, efficient models can run locally on smartphones, laptops, and even IoT devices. 🔍📱 #EdgeAI

2/ Benefits? Enhanced privacy—your data stays on your device. Reduced latency for real-time responses. And increased resilience—no internet? No problem. This democratizes AI, putting powerful tools directly into users' hands. 🌐🔒 #Privacy #Decentralization

3/ Major players are investing heavily in this space. Companies like Meta, Google, and startups are developing lightweight models tailored for consumer hardware. Open-source projects like Llama2 and GPT-4 derivatives are fueling innovation. 🚀🤖 #OpenAI #OpenSource

4/ Challenges remain—balancing model size with accuracy, managing hardware limitations, and ensuring security. But rapid progress suggests these hurdles are surmountable. Expect more capable, personalized AI experiences in everyday devices soon. 🔧⚙️ #AIChallenges

5/ The rise of local AI models signals a shift towards more private, accessible, and efficient AI. As this trend accelerates, we’ll see smarter devices, empowered users, and a new era of AI integration in daily life. Exciting times ahead! 🌟 #FutureOfAI #TechTrends

---

V1 Tweet Score: 44/50


**V2 — Role + Task + Format Tweet**

(1/5) The AI revolution is shifting gears! We're now seeing powerful local AI models running directly on consumer hardware—no cloud needed. This democratizes AI, making it accessible, private, and faster. 🚀 #AI #EdgeComputing

(2/5) Why does this matter? Local AI reduces latency, enhances privacy, and cuts reliance on internet connectivity. Imagine voice assistants that understand you better or personalized AI tools, all on your device. 🔒🤖 #PrivacyFirst

(3/5) Advances in model compression and efficient architectures like GPT-4-tiny and LLaMA are making this possible. Smaller, smarter models can now run on GPUs, smartphones, and even Raspberry Pi! 📱🖥️ #AIonDevice

(4/5) But challenges remain—balancing model size, performance, and energy consumption is key. As hardware improves, expect more sophisticated AI to live right in your pocket, not in the cloud. ⚡ #TechTrends

(5/5) Are we heading toward a future where AI is truly personal and private? What applications excite you most about local AI models? Share your thoughts! 🤔🔍 #FutureOfAI #EdgeAI

---

...

V3 Tweet Score: 43/50


**V4 — With Few-Shot Example Tweet**

(1/5) The rise of local AI models running directly on consumer hardware is transforming privacy, speed, and accessibility. No more reliance on cloud servers—your data stays with you. #AI #EdgeComputing

(2/5) Advances in lightweight architectures like GPT-4 Turbo and LLaMA mean powerful AI can now run on personal laptops, smartphones, and even embedded devices. The future is decentralized! 

(3/5) This shift empowers users with faster responses, enhanced privacy, and reduced costs—no need for constant internet connection or expensive cloud subscriptions. 

(4/5) Developers can now innovate locally, creating customized AI solutions tailored to individual needs without sacrificing control or security. The possibilities are endless! 

(5/5) Are we heading toward a future where AI is truly in everyone’s hands? What new applications do you envision with local AI models? Share your thoughts! 🚀 #AIRevolution #FutureTech

---

V4 Tweet Score: 43/50


**V5 — With Negative Example Tweet**

(1/5) The rise of local AI models running directly on consumer hardware is transforming how we interact with technology. No more reliance on cloud servers—privacy, speed, and control are now in your hands. #LocalAI #TechInnovation

(2/5) Advances in model compression and optimization mean powerful AI can now run smoothly on smartphones, laptops, and even edge devices, making AI more accessible than ever before.

(3/5) This shift enhances privacy—your data stays on your device—while reducing latency, enabling real-time responses for applications like AR, voice assistants, and personalized recommendations.

(4/5) Challenges remain in balancing model size, performance, and energy consumption, but ongoing research promises even more efficient local AI solutions in the near future.

(5/5) How will local AI change your daily tech experience? Are you excited to have more control over your data and devices? Share your thoughts! 🚀 #AI #FutureTech

---

V5 Tweet Score: 44/50

--- Tweet Refinement Summary ---
V0 — Vague Tweet                              -> Total: 45/50
V1 — Role + Task Tweet                        -> Total: 44/50
V2 — Role + Task + Format Tweet               -> Total: 44/50
V3 — Role + Task + Format + Constraints Tweet -> Total: 43/50
V4 — With Few-Shot Example Tweet              -> Total: 43/50
V5 — With Negative Example Tweet              -> Total: 44/50

Best Tweet Version:  V0 — Vague Tweet
{
  "relevance": 10,
  "engagement": 8,
  "clarity_conciseness": 8,
  "thread_structure": 9,
  "no_hashtag_spam": 10,
  "total": 45,
  "comments": "Highly relevant to local AI models on consumer hardware | Engaging with current trends and future implications | Clear and concise, well within tweet length | Well-structured with logical flow and numbering | No excessive or irrelevant hashtags present"
}

Worst Tweet Version: V3 — Role + Task + Format + Constraints Tweet
{
  "relevance": 9,
  "engagement": 8,
  "clarity_conciseness

## 🏁 Summary & Key Takeaways

| Prompt Component | What it does | When to add it |
|---|---|---|
| Role | Shifts voice & expertise | Almost always |
| Context | Provides domain knowledge | When task is specialized |
| Task | Defines the job | Always (be specific) |
| Format | Controls structure | When structure matters |
| Constraints | Rules out bad outputs | When you keep seeing specific failures |
| Positive examples | Shows "good" | When format is complex |
| Negative examples | Shows "bad" | When specific errors keep recurring |
| Rubric evaluation | Measures quality | When you need consistency at scale |

### What's next?
- **Lesson 4 Deep Dive**: Chaining Prompts — connect multiple refined prompts in sequence.
